## Enter API related variables for calls

Set these before running the cell below. They come from your own Toast
account and Colab setup, not from this repo:

| Variable | Where to find it |
|---|---|
| `hostname` | Toast API subdomain for your integration, e.g. `ws-api.toasttab.com` (Toast dev portal -> API access) |
| `Client_ID` | Toast API Client ID (Toast dev portal -> your API access group -> Standard API credentials) |
| `Client_Secret` | Toast API Client Secret (same page as Client ID -- only shown once at creation, store it somewhere safe) |
| `Restaurant_GUID` | GUID of the specific restaurant location you're extracting data for (Toast dev portal -> Restaurants) |

In Google Colab: **Runtime -> Secrets** (key icon, left sidebar) -> add each
value under the exact name shown above -> grant this notebook access when
prompted. `userdata.get(...)` reads them from there. Never hardcode real
credentials directly into a cell.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
userdata.get('secretName')

import requests
import json
import time
import os
import getpass


# OUTPUT DIR
OUTPUT_DIR = "/content/drive/MyDrive/Data Projects/restaurant-analytics/catalogs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Getting Toast API hostname (e.g. ws-api.toasttab.com):")
TOAST_HOSTNAME = userdata.get('hostname').strip()
TOAST_HOSTNAME = TOAST_HOSTNAME.replace("https://", "").replace("http://", "").rstrip("/")

print("Getting Toast API Client ID:")
CLIENT_ID = userdata.get('Client_ID').strip()

print("Getting Toast API Client Secret:")
CLIENT_SECRET = userdata.get('Client_Secret').strip()

print("Getting Toast API restaurant GUID:")
RESTAURANT_GUID = userdata.get('Restaurant_GUID').strip()

In [ ]:
def get_access_token():
    """
    Logins to Toast and asks for an Access Token
    Args:
      None
    Output:
      access_token, expires_at_timestamp
    """
    url = f"https://{TOAST_HOSTNAME}/authentication/v1/authentication/login"
    body = {"clientId": CLIENT_ID,
            "clientSecret": CLIENT_SECRET,
            "userAccessType": "TOAST_MACHINE_CLIENT"
            }

    # Troubleshooting
    print(f"URL: {url}")
    print(f"Client ID (repr): {repr(CLIENT_ID)}  | length: {len(CLIENT_ID)}")
    print(f"Client Secret (repr, primeros/últimos 3 chars): "
          f"{repr(CLIENT_SECRET[:3])}...{repr(CLIENT_SECRET[-3:])} | length: {len(CLIENT_SECRET)}")
    print(f"userAccessType (repr): {repr(body['userAccessType'])}")
    response = requests.post(url, json=body)

    if response.status_code != 200:
        print(f"ERROR {response.status_code}.")
        print(f"Server Response: {response.text}")

    response.raise_for_status()
    data = response.json()

    print("LOGIN RESPONSE SUCCESSFUL (full structure):......")
    print(json.dumps(data, indent=2))

    token_block = data["token"]
    access_token = token_block["accessToken"]
    expires_in = token_block["expiresIn"]  # seconds
    expires_at = time.time() + expires_in

    print(f"Obtained Token expired in {expires_in} segundos (~{expires_in/3600:.1f} hours).")
    return access_token, expires_at


def ensure_valid_token(access_token, expires_at):
    """
    Refreshes the token if it has less than 60 seconds of life remaining.
    """
    if time.time() > (expires_at - 60):
        print("Token about to expire, requesting a new one...")
        return get_access_token()
    return access_token, expires_at

In [ ]:
CATALOGS = [# (endpoint, output_filename, label)
           ("/config/v2/salesCategories", "catalog_sales_categories.json", "salesCategories"),
           ("/config/v2/revenueCenters", "catalog_revenue_centers.json", "revenueCenters"),
           ("/config/v2/diningOptions", "catalog_dining_options.json", "diningOptions"),
           ("/config/v2/tables", "catalog_tables.json", "tables"),
           ("/config/v2/menuItems", "catalog_menu_items.json", "menuItems"),
           ("/labor/v1/employees", "catalog_employees.json", "employees"),
           ("/labor/v1/jobs", "catalog_jobs.json", "jobs")
            ]

SLEEP_BETWEEN_CATALOGS = 2.0  # seconds between calls (catalogs are small, no bulk rate limit needed)


def fetch_catalog(endpoint, access_token, restaurant_guid, label):
    """
    Calls a Configuration API or Labor API endpoint that returns a list of records.
    Args:
        endpoint: API path, e.g. "/config/v2/salesCategories"
        access_token: Bearer token
        restaurant_guid: Restaurant context header value
        label: Human-readable name for logs, e.g. "salesCategories"

    Returns:
        List of all records across all pages, or partial/empty list on failure.
    """
    base_url = f"https://{TOAST_HOSTNAME}{endpoint}"
    headers = {"Authorization": f"Bearer {access_token}",
               "Toast-Restaurant-External-ID": restaurant_guid}

    MAX_RETRIES = 3
    all_records = []
    page_token = None
    page_num = 1

    while True:
      params = {"pageToken": page_token} if page_token else {}
      retry_count = 0
      page_succeeded = False
      data = None
      response = None

      while retry_count < MAX_RETRIES:
        response = requests.get(base_url, headers=headers, params=params)

        if response.status_code == 429:
            print(f"  [{label}] Rate limited (page {page_num}). Waiting 15 seconds...")
            time.sleep(15)
            continue  # retry without incrementing retry_count

        if response.status_code == 200:
            data = response.json()
            page_succeeded = True
            break

        retry_count += 1
        print(f"  [{label}] ERROR {response.status_code} on page {page_num} "
              f"(attempt {retry_count}/{MAX_RETRIES}).")
        print(f"  URL: {response.url}")
        print(f"  Server response: {response.text}")

        if response.status_code >= 500 and retry_count < MAX_RETRIES:
            wait = 5 * retry_count
            print(f"  Retrying in {wait} seconds...")
            time.sleep(wait)
        else:
            break  # 4xx: no point retrying

      if not page_succeeded:
          print(f"  [{label}] Failed on page {page_num} after {retry_count} attempts. "
                f"Returning {len(all_records)} records collected so far.")
          return all_records

      all_records.extend(data)
      print(f"  [{label}] Page {page_num}: {len(data)} records "
            f"(total so far: {len(all_records)}).")

      page_token = response.headers.get("Toast-Next-Page-Token")
      if not page_token:
          break  # no more pages

      page_num += 1
      time.sleep(0.5)  # small pause between pages

    print(f"  [{label}] OK {len(all_records)} total records across {page_num} page(s).")
    return all_records

def run_catalog_extraction():
    """
    Extracts all catalogs needed to resolve GUIDs to human-readable names.
    Each catalog is saved as a JSON file in OUTPUT_DIR.

    Catalogs extracted:
        - salesCategories: resolves sales_category_guid
        - revenueCenters: resolves revenue_center_guid
        - diningOptions: resolves dining_option_guid
        - tables: resolves table_guid
        - menuItems: resolves item_guid (also includes sku, plu, calories, visibility)
        - employees: resolves server_guid
    """
    access_token, expires_at = get_access_token()

    for endpoint, filename, label in CATALOGS:
        output_path = os.path.join(OUTPUT_DIR, filename)

        print(f"\nExtracting [{label}].........")

        access_token, expires_at = ensure_valid_token(access_token, expires_at)
        records = fetch_catalog(endpoint, access_token, RESTAURANT_GUID, label)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)

        print(f"  [{label}] Saved: {len(records)} records: {output_path}")
        time.sleep(SLEEP_BETWEEN_CATALOGS)
    print("Catalog extraction complete!!!")

In [ ]:
ITEM_GROUP_GUIDS_LIST = []  # e.g. df['item_group_guid'].dropna().unique().tolist()

def fetch_menu_groups_by_guid(item_group_guids_list, access_token, restaurant_guid):
    """
    Unlike the other catalogs, Toast has no "list all menu groups" endpoint.
    Each MenuGroup must be fetched individually via GET /menuGroups/{guid}.

    Args:
        item_group_guids_list: list of distinct item_group_guid values found in your order data
                               (get these from df['item_group_guid'].dropna().unique().tolist())
        access_token, restaurant_guid: same as other catalog calls

    Returns:
        List of dicts: [{"guid": "...", "name": "...", ...}, ...]
        Skips (and logs) any GUID that fails to resolve.
    """
    headers = {"Authorization": f"Bearer {access_token}",
               "Toast-Restaurant-External-ID": restaurant_guid
               }

    resolved_groups = []
    failed_guids = []

    print(f"\nResolving {len(item_group_guids_list)} menu group GUIDs individually "
          f"(no bulk-list endpoint exists for menuGroups)...")

    # LOOP through the GUIDs of MenuGrops
    for idx, guid in enumerate(item_group_guids_list, start=1):
        url = f"https://{TOAST_HOSTNAME}/config/v2/menuGroups/{guid}"
        response = requests.get(url, headers=headers)

        if response.status_code == 429:
            print(f"  [{idx}/{len(item_group_guids_list)}] Rate limited. Waiting 15 seconds...")
            time.sleep(15)
            response = requests.get(url, headers=headers)  # single retry after wait

        if response.status_code == 200:
            resolved_groups.append(response.json())
        else:
            print(f"  [{idx}/{len(item_group_guids_list)}] WARNING: failed to resolve {guid} "
                  f"(status {response.status_code})")
            failed_guids.append(guid)

        time.sleep(0.3)

    print(f"Resolved {len(resolved_groups)}/{len(item_group_guids_list)} menu groups. "
          f"{len(failed_guids)} failed.")

    return resolved_groups, failed_guids


def run_menu_groups_extraction(item_group_guids_list):
    """
    Resolves a list of item_group_guid values (e.g. from your orders DataFrame)
    to their MenuGroup names, and saves the result as catalog_menu_groups.json.

    This is separate from run_catalog_extraction() because menuGroups requires
    one API call per GUID (no bulk-list endpoint), unlike the other catalogs.

    Args:
        item_group_guids_list: list of distinct GUIDs to resolve.
                              Get these from your orders DataFrame, e.g.:
                              df['item_group_guid'].dropna().unique().tolist()
    """
    access_token, expires_at = get_access_token()
    access_token, expires_at = ensure_valid_token(access_token, expires_at)

    resolved_groups, failed_guids = fetch_menu_groups_by_guid(item_group_guids_list,
                                                              access_token,
                                                              RESTAURANT_GUID)

    output_path = os.path.join(OUTPUT_DIR, "catalog_menu_groups.json")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(resolved_groups, f, ensure_ascii=False, indent=2)
    print(f"Saved: {len(resolved_groups)} menu groups → {output_path}")

    if failed_guids:
      failed_path = os.path.join(OUTPUT_DIR, "FAILED_menu_group_guids.json")
      with open(failed_path, "w", encoding="utf-8") as f:
          json.dump({"failed_guids": failed_guids}, f, indent=2)
      print(f"****** {len(failed_guids)} GUIDs failed to resolve. "
            f"Logged to {failed_path}")

    return resolved_groups, failed_guids

In [ ]:
run_catalog_extraction()

In [ ]:
# menu_groups = run_menu_groups_extraction(ITEM_GROUP_GUIDS_LIST)